In [1]:
# Dependencies
import pandas as pd
import os
import re

# Base Directory
base_dir = os.path.abspath("/Users/amberteetsel/MSDS/NLP/nlp-author-identification/")

In [2]:
# Remove Gutenberg Header and Footer
def remove_gutenberg(text):
    """Strips text outside Gutenberg header and footer"""
    start_txt = re.search(r"\*\*\* START OF .*?\*\*\*", text)
    end_txt = re.search(r"\*\*\* END OF .*?\*\*\*", text)
    start = start_txt.end() if start_txt else 0
    end = end_txt.start() if end_txt else len(text)
    return text[start:end].strip()

# should also strip author names, chapters, table of contents, etc.
def remove_front_extra(text):
    """Removes everything before first chapter heading"""
    ch1 = re.search(r"\n\s*(CHAPTER|Chapter)\s+(I|1|ONE|One)\b", text)
    if ch1:
        return text[ch1.end():].strip()
    else:
        return text

def remove_chapter_headings(text):
    """Removes 'Chapter I', "CHAPTER IX", etc."""
    pattern = r"^\s*(CHAPTER|Chapter)\s+[IVXLCDM]+\.?\s*$"
    return re.sub(pattern, "", text, flags=re.MULTILINE)

def clean_text(inpath, outpath):
    """Cleans text files and saves to a new file"""
    with open(inpath, encoding="utf-8") as f:
        text = f.read()
    text = remove_gutenberg(text)
    text = remove_front_extra(text)
    text = remove_chapter_headings(text)
    text = re.sub(r"\n{3,}", "\n\n", text) # remove excess newlines
    with open(outpath, "w", encoding="utf-8") as f:
        f.write(text)
    return text

In [3]:
# Clean raw files and save
hobbit_input_path = os.path.join(base_dir, "texts", "hobbit_raw.txt")
hobbit_output_path = os.path.join(base_dir, "texts", "hobbit_clean.txt")
hobbit = clean_text(hobbit_input_path, hobbit_output_path)

lw_input_path = os.path.join(base_dir, "texts", "lostworld_raw.txt")
low_output_path = os.path.join(base_dir, "texts", "lostworld_clean.txt")
lostworld = clean_text(lw_input_path, low_output_path)

## Train-Test Split

Splitting by sentences instead of paragraphs due to inconsistent nature of newlines in source text (they can occur mid-sentence, up to 3-4 newlines). Deliberately simplified for purposes of this analysis.

In [4]:
# AI Code
def split_train_holdout(text, holdout_frac=0.1):
    # Collapse ALL whitespace/newlines into single spaces —
    # don't trust newline count to mean anything structurally
    flat = re.sub(r"\s+", " ", text).strip()
    
    # Split into sentences: break after ., !, or ? followed by a space + capital letter
    sentences = re.split(r"(?<=[.!?])\s+(?=[A-Z])", flat)
    
    n_holdout = max(1, int(len(sentences) * holdout_frac))
    
    train = sentences[:-n_holdout]
    holdout = sentences[-n_holdout:]
    
    return " ".join(train), " ".join(holdout)

In [5]:
hobbit_train, hobbit_holdout = split_train_holdout(hobbit)
lostworld_train, lostworld_holdout = split_train_holdout(lostworld)

In [6]:
# Save to files
hobbit_train_path = os.path.join(base_dir, "texts", "hobbit_train.txt")
hobbit_holdout_path = os.path.join(base_dir, "texts", "hobbit_holdout.txt")
lostworld_train_path = os.path.join(base_dir, "texts", "lostworld_train.txt")
lostworld_holdout_path = os.path.join(base_dir, "texts", "lostworld_holdout.txt")
paths = [hobbit_train_path, hobbit_holdout_path, lostworld_train_path, lostworld_holdout_path]

for path in paths:
    with open(path, "w", encoding="utf-8") as f:
        if "hobbit" in path:
            f.write(hobbit_train if "train" in path else hobbit_holdout)
        else:
            f.write(lostworld_train if "train" in path else lostworld_holdout)

In [7]:
def stats(text, label):
    n_chars = len(text)
    n_words = len(text.split())
    n_sentences = len(re.split(r"(?<=[.!?])\s+(?=[A-Z])", text))
    print(f"--- {label} ---")
    print(f"chars:     {n_chars:,}")
    print(f"words:     {n_words:,}")
    print(f"sentences: {n_sentences:,}")
    print()

def sanity_check(book_name, train_text, holdout_text):
    print(f"=== {book_name} ===")
    stats(train_text, "train")
    stats(holdout_text, "holdout")

    total_words = len(train_text.split()) + len(holdout_text.split())
    holdout_pct = len(holdout_text.split()) / total_words * 100
    print(f"holdout = {holdout_pct:.1f}% of total words (target ~10%)\n")

    # eyeball the boundary
    print("Last 200 chars of TRAIN:")
    print(repr(train_text[-200:]))
    print()
    print("First 200 chars of HOLDOUT:")
    print(repr(holdout_text[:200]))
    print("=" * 60)
    print()

In [8]:
sanity_check("Hobbit", hobbit_train, hobbit_holdout)

=== Hobbit ===
--- train ---
chars:     459,506
words:     87,060
sentences: 3,568

--- holdout ---
chars:     47,064
words:     8,837
sentences: 396

holdout = 9.2% of total words (target ~10%)

Last 200 chars of TRAIN:
'ge. “I gave it to them!” squeaked Bilbo, who was peering over the wall, by now in a dreadful fright. “You! You!” cried Thorin, turning upon him and grasping him with both hands. “You miserable hobbit!'

First 200 chars of HOLDOUT:
'You undersized — burglar!” he shouted at a loss for words, and he shook poor Bilbo like a rabbit. “By the beard of Durin! I wish I had Gandalf here! Curse him for his choice of you! May his beard with'



In [9]:
sanity_check("Lost World", lostworld_train, lostworld_holdout)

=== Lost World ===
--- train ---
chars:     374,158
words:     67,943
sentences: 2,788

--- holdout ---
chars:     43,580
words:     7,685
sentences: 309

holdout = 10.2% of total words (target ~10%)

Last 200 chars of TRAIN:
' of the most elementary developments of man." "It is clearly some sort of script," said Challenger. "Looks like a guinea puzzle competition," remarked Lord John, craning his neck to have a look at it.'

First 200 chars of HOLDOUT:
'Then suddenly he stretched out his hand and seized the puzzle. "By George!" he cried, "I believe I\'ve got it. The boy guessed right the very first time. See here! How many marks are on that paper? Eig'



In [10]:
with open(hobbit_input_path, 'r', encoding='utf-8') as f:
    hobbit_raw = f.read()
    print(repr(hobbit_raw[:500]))

'THE HOBBIT \n\nOR \n\nTHERE AND BACK \nAGAIN \n\n\n\nJ.R.R. TOLKIEN \n\nChapter I \n\nAN UNEXPECTED PARTY \n\nIn a hole in the ground there lived a hobbit. Not a nasty, dirty, wet\nhole, filled with the ends of worms and an oozy smell, nor yet a dry,\nbare, sandy hole with nothing in it to sit down on or to eat: it was a\nhobbit-hole, and that means comfort.\n\nIt had a perfectly round door like a porthole, painted green, with a\nshiny yellow brass knob in the exact middle. The door opened on to a\ntube-shaped hall l'


In [11]:
with open(lw_input_path, 'r', encoding='utf-8') as f:
    lw_raw = f.read()
    print(repr(lw_raw[:500]))

'The Project Gutenberg eBook of The Lost World\n    \nThis eBook is for the use of anyone anywhere in the United States and\nmost other parts of the world at no cost and with almost no restrictions\nwhatsoever. You may copy it, give it away or re-use it under the terms\nof the Project Gutenberg License included with this eBook or online\nat www.gutenberg.org. If you are not located in the United States,\nyou will have to check the laws of the country where you are located\nbefore using this eBook.\n\nTitle'
